# Step 5 — Yearly citations: corrected standalone collector

Open this notebook from inside the repository, restart the kernel, and Run All. No repair script and no execution of the original notebook are required.

## What is preserved
- Reads steps 2 and 4 without modifying them.
- Uses your existing `yearly_citations_v1/citation_collection.sqlite` checkpoint and request ledger.
- Makes a one-time checkpoint backup before migration. Completed portfolio downloads remain reusable.
- Old citation queries/edges are preserved but marked superseded and excluded from the corrected totals.
- Imports successful diagnostic ID mappings if the saved JSON exists. The two user-reported 404s remain explicit coverage gaps.

## Correct measure
A citation is a distinct (citing work, canonical cited portfolio work) link. Bin links by the citing work's publication year. A citing paper referencing two different works by one author contributes two citations. Multiple old aliases for the same target count once. Include all known portfolio works, not just works published near the award. Self-citations are included for step 6 to classify. Award year is separate from pre/post totals.

## Collection and limitations
First finish any portfolio refresh; then batch-check all target IDs, resolving ambiguous IDs individually; then collect citing works. Budget stops are normal and resumable. Several runs may do only ID auditing before citation collection resumes. Default: 200 HTTP attempts per run, 5,000 per UTC day through this collector. These are LOCAL caps, not account-wide budget monitoring; check your dashboard. Full collection may require several days.

401/403/429 stop immediately. Server failures receive bounded backoff. Changed query totals receive at most two archived fresh-query retries. Other validation failures stay blocked for review. Missing/unresolved targets are not treated as zeros.

Citations are reconstructed from the current indexed graph, not historical index snapshots. Legacy portfolio completeness and author disambiguation remain unverified. Do not claim final completeness merely because a query finishes.

Credentials: read `OPENALEX_API_KEY` or use the masked prompt. No credentials are embedded or saved. Do not run the old and corrected collectors simultaneously against the same database.

In [7]:
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict, Counter
import getpass, hashlib, io, json, math, os, pickle, re, sqlite3, time
import pandas as pd
import requests
from IPython.display import display

ROOT_OVERRIDE = None
RUN_API_COLLECTION = True
MAX_REQUESTS_PER_RUN = 5000
MAX_REQUESTS_PER_UTC_DAY = 30000
PER_PAGE = 100
BATCH_SIZE = 100
REQUEST_INTERVAL_SECONDS = 0.2
CORPUS = 'all'
COLLECTION_VERSION = 'yearly_citations_v1'
ALIAS_POLICY = 'canonical_alias_v2'
STEP4_REL = 'conf_paper/conf_data/derived/yearly_publications_v1'
STEP2_REL = 'conf_paper/conf_data/derived/reference_awards_v1'
CACHE_REL = 'conf_paper/conf_data/04_all_works_cache.pkl'
OFFSETS = list(range(-5, 6))

def now(): return datetime.now(timezone.utc).isoformat()
def day(): return datetime.now(timezone.utc).date().isoformat()
def sha(p): return hashlib.sha256(p.read_bytes()).hexdigest()
def stable_hash(v): return hashlib.sha256(json.dumps(v,sort_keys=True,separators=(',',':')).encode()).hexdigest()
def csv(p): return pd.read_csv(p,dtype=str,keep_default_na=False,encoding='utf-8-sig')
def nid(v,prefix='W'):
    if not isinstance(v,str): return None
    m=re.fullmatch(r'(?:https?://openalex\.org/)?('+prefix+r'\d+)/?',v.strip(),re.I)
    return m.group(1).upper() if m else None

def yr(v):
    if isinstance(v,bool): return None
    try:
        n=float(v)
        return int(n) if math.isfinite(n) and n.is_integer() and 1<=n<=datetime.now(timezone.utc).year else None
    except (ValueError,TypeError,OverflowError): return None

def verify(folder):
    meta=json.loads((folder/'manifest.json').read_text(encoding='utf-8'))
    for name,h in meta['output_sha256'].items():
        if not (folder/name).is_file() or sha(folder/name)!=h:
            raise RuntimeError('Input integrity failure: '+str(folder/name))
    return meta

candidates=[Path(ROOT_OVERRIDE).expanduser().resolve()] if ROOT_OVERRIDE else [Path.cwd().resolve(),*Path.cwd().resolve().parents]
ROOT=next((p for p in candidates if (p/STEP4_REL/'manifest.json').is_file()),None)
if ROOT is None: raise FileNotFoundError('Run inside the repository or set ROOT_OVERRIDE.')
STEP4=ROOT/STEP4_REL; STEP2=ROOT/STEP2_REL
m4=verify(STEP4); m2=verify(STEP2)
if sha(STEP2/'manifest.json')!=m4['identity']['input_hashes']['step2_manifest']:
    raise RuntimeError('Step 4 references different step-2 inputs.')
master4=csv(STEP4/'author_yearly_publications.csv')
links=csv(STEP2/'reference_year_award_links.csv')
review4=csv(STEP4/'cache_review.csv')
assert master4.author_id.is_unique and master4.author_id.map(lambda x:nid(x,'A')).eq(master4.author_id).all()
master4['reference_award_year']=master4.reference_award_year.map(yr)
assert master4.reference_award_year.notna().all() and master4.reference_award_year.between(2000,2018).all()
WINDOW_LO=int(master4.reference_award_year.min())-5
WINDOW_HI=int(master4.reference_award_year.max())+5

class BuiltinsOnlyUnpickler(pickle.Unpickler):
    def find_class(self,module,name): raise pickle.UnpicklingError('Global/class loading forbidden.')
    def persistent_load(self,pid): raise pickle.UnpicklingError('Persistent loading forbidden.')
cache_path=ROOT/CACHE_REL
if sha(cache_path)!=m4['identity']['input_hashes']['works_cache']:
    raise RuntimeError('Works cache changed since step 4.')
cache=BuiltinsOnlyUnpickler(io.BytesIO(cache_path.read_bytes())).load()
if not isinstance(cache,dict) or not isinstance(cache.get('all_works'),dict): raise ValueError('Unexpected cache schema.')
legacy=defaultdict(set); legacy_bad=set()
for raw_aid,works in cache['all_works'].items():
    aid=nid(raw_aid,'A')
    if not aid: continue
    if not isinstance(works,list): legacy_bad.add(aid); continue
    for w in works:
        if isinstance(w,dict) and nid(w.get('id')): legacy[aid].add(nid(w['id']))
        else: legacy_bad.add(aid)
expected_refs=defaultdict(set)
for r in links.itertuples(index=False):
    if nid(r.openalex_work_id): expected_refs[r.author_id].add(nid(r.openalex_work_id))
refresh_ids=(set(review4.author_id)|legacy_bad|(set(master4.author_id)-set(legacy)))&set(master4.author_id)
inputs={'step4_manifest':sha(STEP4/'manifest.json'),'step2_manifest':sha(STEP2/'manifest.json'),'works_cache':sha(cache_path)}
print('Authors:',len(master4),'| Window:',WINDOW_LO,'through',WINDOW_HI,'| Portfolios queued:',len(refresh_ids))

Authors: 2710 | Window: 1995 through 2023 | Portfolios queued: 125


## 1. Open and back up the existing checkpoint

Completed portfolio tasks keep their original IDs. Old citation tasks are superseded, not deleted. If the prior repair was already run, its alias table and backup are reused.

In [8]:
STATE_DIR=ROOT/'conf_paper/conf_data/collection'/COLLECTION_VERSION
STATE_DIR.mkdir(parents=True,exist_ok=True)
DB_PATH=STATE_DIR/'citation_collection.sqlite'
if 'conn' in globals():
    try: conn.close()
    except Exception: pass
existed=DB_PATH.is_file()
conn=sqlite3.connect(DB_PATH,timeout=30); conn.row_factory=sqlite3.Row
had_metadata=conn.execute("SELECT 1 FROM sqlite_master WHERE type='table' AND name='metadata'").fetchone() is not None
had_backup=conn.execute("SELECT value FROM metadata WHERE key='alias_repair_backup_v2'").fetchone() if had_metadata else None
backup_name=None
if existed and not had_backup:
    backup_name='citation_collection_before_alias_v2_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')+'.sqlite'
    backup=sqlite3.connect(DB_PATH.with_name(backup_name))
    try: conn.backup(backup)
    finally: backup.close()

conn.executescript("""
CREATE TABLE IF NOT EXISTS metadata(key TEXT PRIMARY KEY,value TEXT NOT NULL);
CREATE TABLE IF NOT EXISTS tasks(id TEXT PRIMARY KEY,kind TEXT NOT NULL,payload TEXT NOT NULL,status TEXT NOT NULL DEFAULT 'pending',cursor TEXT NOT NULL DEFAULT '*',expected INTEGER,pages INTEGER NOT NULL DEFAULT 0,issue TEXT NOT NULL DEFAULT '',started TEXT,finished TEXT);
CREATE TABLE IF NOT EXISTS items(task TEXT NOT NULL,id TEXT NOT NULL,year INTEGER,author_ids TEXT NOT NULL,PRIMARY KEY(task,id));
CREATE INDEX IF NOT EXISTS items_id_year ON items(id,year);
CREATE TABLE IF NOT EXISTS edges(task TEXT NOT NULL,citing TEXT NOT NULL,cited TEXT NOT NULL,year INTEGER NOT NULL,PRIMARY KEY(task,citing,cited));
CREATE INDEX IF NOT EXISTS edge_cited_year ON edges(cited,year);
CREATE TABLE IF NOT EXISTS request_ledger(utc_day TEXT NOT NULL,at TEXT NOT NULL,status TEXT);
CREATE TABLE IF NOT EXISTS work_aliases_v2(old_id TEXT PRIMARY KEY,canonical_id TEXT,status TEXT NOT NULL,source TEXT NOT NULL,checked_at TEXT NOT NULL);
CREATE TABLE IF NOT EXISTS query_restarts_v2(restart_id INTEGER PRIMARY KEY,task TEXT NOT NULL,reason TEXT NOT NULL,at TEXT NOT NULL);
CREATE TABLE IF NOT EXISTS retired_items_v2(restart_id INTEGER,task TEXT,id TEXT,year INTEGER,author_ids TEXT);
CREATE TABLE IF NOT EXISTS retired_edges_v2(restart_id INTEGER,task TEXT,citing TEXT,cited TEXT,year INTEGER);
CREATE TEMP TABLE IF NOT EXISTS active_citation_tasks_v2(id TEXT PRIMARY KEY);
""")
# Retain legacy identity exactly for checkpoint compatibility. Repair policy is recorded separately.
identity={'inputs':inputs,'transform_version':'yearly_citations_1.0','corpus':CORPUS,
          'batch_size':BATCH_SIZE,'per_page':PER_PAGE,'window':[WINDOW_LO,WINDOW_HI]}
prior=conn.execute("SELECT value FROM metadata WHERE key='identity'").fetchone()
if prior and json.loads(prior['value'])!=identity:
    raise RuntimeError('Checkpoint inputs/settings differ. Do not mix collections.')
with conn:
    conn.execute("INSERT OR IGNORE INTO metadata VALUES('identity',?)",(json.dumps(identity,sort_keys=True),))
    if backup_name: conn.execute("INSERT OR IGNORE INTO metadata VALUES('alias_repair_backup_v2',?)",(backup_name,))
    for row in conn.execute("SELECT id,payload FROM tasks WHERE kind='citations'").fetchall():
        if json.loads(row['payload']).get('alias_policy')!=ALIAS_POLICY:
            conn.execute("UPDATE tasks SET status='superseded' WHERE id=?",(row['id'],))

alias_file=STATE_DIR/'target_alias_diagnostic.json'
if alias_file.exists():
    saved=json.loads(alias_file.read_text(encoding='utf-8'))
    if not isinstance(saved,dict): raise ValueError('Unexpected diagnostic alias format.')
    with conn:
        for old,canonical in saved.items():
            if nid(old) and nid(canonical):
                conn.execute('INSERT OR IGNORE INTO work_aliases_v2 VALUES(?,?,?,?,?)',
                    (nid(old),nid(canonical),'resolved','saved_diagnostic_lookup',now()))
with conn:
    for old in ['W101060335','W101505544']:
        conn.execute('INSERT OR IGNORE INTO work_aliases_v2 VALUES(?,?,?,?,?)',
            (old,None,'unresolved_404','user_reported_diagnostic_404',now()))

def add_task(kind,payload):
    task_id=stable_hash({'kind':kind,'payload':payload,'corpus':CORPUS})
    with conn:
        conn.execute('INSERT OR IGNORE INTO tasks(id,kind,payload,status) VALUES(?,?,?,?)',
            (task_id,kind,json.dumps(payload,sort_keys=True),'pending'))
    return task_id
portfolio_tasks={aid:add_task('portfolio',{'author_id':aid}) for aid in sorted(refresh_ids)}
print('Checkpoint ready. Existing completed portfolios will be reused.')

Checkpoint ready. Existing completed portfolios will be reused.


## 2. Budgeted requests, retries and canonical target-ID auditing

Batch responses only confirm requested IDs that return unchanged. Missing IDs are resolved individually; returned records are never paired to old IDs by list order. A 404 is recorded as unresolved, not a zero. ID auditing is checkpointed and may consume the first several capped runs.

In [9]:
class StopCollection(Exception): pass
class BadPage(Exception): pass
session=requests.Session()
requests_this_run=0
stop_reason='not_started'
ALIAS_MAP={}

def get_api_key():
    key=os.environ.get('OPENALEX_API_KEY','').strip()
    if not key: key=getpass.getpass('OpenAlex API key (hidden, not saved): ').strip()
    if not key: raise StopCollection('No API key supplied.')
    return key

def request_json(path,params=None,allow_404=False):
    global requests_this_run
    for attempt in range(4):
        used=conn.execute('SELECT COUNT(*) AS n FROM request_ledger WHERE utc_day=?',(day(),)).fetchone()['n']
        if requests_this_run>=MAX_REQUESTS_PER_RUN: raise StopCollection('Per-run request cap reached; resume next run.')
        if used>=MAX_REQUESTS_PER_UTC_DAY: raise StopCollection('Local UTC-day cap reached; resume after reset.')
        with conn:
            c=conn.execute('INSERT INTO request_ledger VALUES(?,?,?)',(day(),now(),'attempted'))
            ledger_id=c.lastrowid
        requests_this_run+=1
        time.sleep(REQUEST_INTERVAL_SECONDS)
        try: response=session.get('https://api.openalex.org/'+path,params=params,timeout=(15,120))
        except requests.RequestException:
            if attempt==3: raise StopCollection('Network failure; progress saved. Credentials not logged.')
            time.sleep(min(30,2**(attempt+1))); continue
        status=response.status_code
        with conn: conn.execute('UPDATE request_ledger SET status=? WHERE rowid=?',(str(status),ledger_id))
        if status==404 and allow_404: return None
        if status in (401,403,429): raise StopCollection('HTTP '+str(status)+': check key/budget/rate limit; progress saved.')
        if status>=500:
            if attempt==3: raise StopCollection('Repeated HTTP '+str(status)+'; resume later.')
            time.sleep(min(30,2**(attempt+1))); continue
        if status!=200: raise StopCollection('HTTP '+str(status)+': inspect query compatibility; no response body logged.')
        try: data=response.json()
        except ValueError: raise StopCollection('Invalid JSON; cursor unchanged.')
        if not isinstance(data,dict): raise StopCollection('Unexpected JSON type.')
        return data
    raise StopCollection('Retry limit reached.')

def reload_aliases():
    global ALIAS_MAP
    ALIAS_MAP={r['old_id']:r['canonical_id'] for r in conn.execute("SELECT old_id,canonical_id FROM work_aliases_v2 WHERE status='resolved'")}

def canonical(w):
    w=nid(w); seen=set()
    while w in ALIAS_MAP and ALIAS_MAP[w]!=w:
        if w in seen: raise RuntimeError('Alias cycle; inspect work_alias_audit.csv.')
        seen.add(w); w=ALIAS_MAP[w]
    return w

def resolve_single(old):
    data=request_json('works/'+old,{'corpus':CORPUS},allow_404=True)
    current=nid(data.get('id')) if data is not None else None
    if data is not None and not current: raise StopCollection('Work lookup has no valid ID.')
    with conn:
        conn.execute('INSERT OR REPLACE INTO work_aliases_v2 VALUES(?,?,?,?,?)',
            (old,current,'resolved' if current else 'unresolved_404','single_record_lookup',now()))

def resolve_targets(targets):
    targets=set(targets)
    states={r['old_id']:r['status'] for r in conn.execute('SELECT old_id,status FROM work_aliases_v2')}
    for old in sorted(targets):
        if states.get(old)=='pending_single': resolve_single(old)
    while True:
        states={r['old_id']:r['status'] for r in conn.execute('SELECT old_id,status FROM work_aliases_v2')}
        pending=sorted(targets-set(states))
        if not pending: break
        batch=pending[:BATCH_SIZE]
        data=request_json('works',{'filter':'ids.openalex:'+'|'.join(batch),'select':'id','per_page':PER_PAGE,'corpus':CORPUS})
        results=data.get('results'); total=data.get('meta',{}).get('count')
        if not isinstance(results,list) or isinstance(total,bool) or not isinstance(total,int) or total!=len(results) or not 0<=total<=len(batch):
            raise StopCollection('Incomplete/inconsistent ID-batch lookup; no aliases guessed.')
        returned=[nid(r.get('id')) if isinstance(r,dict) else None for r in results]
        if any(x is None for x in returned) or len(set(returned))!=len(returned):
            raise StopCollection('Invalid/duplicate IDs in batch lookup.')
        direct=set(returned)&set(batch)
        with conn:
            for old in batch:
                conn.execute('INSERT OR IGNORE INTO work_aliases_v2 VALUES(?,?,?,?,?)',
                    (old,old if old in direct else None,'resolved' if old in direct else 'pending_single','batch_lookup',now()))
        for old in batch:
            if old not in direct: resolve_single(old)
    reload_aliases()
reload_aliases()

## 3. Page validation and bounded fresh-query restarts

A changed total does not silently pass: the affected partial query is archived and restarted, at most twice. Other validation failures remain blocked. No citation page is saved unless all its records pass validation.

In [ ]:
def validate_page(task,body):
    p=json.loads(task['payload']); meta=body.get('meta'); results=body.get('results')
    if not isinstance(meta,dict) or not isinstance(results,list): raise BadPage('missing_meta_or_results')
    total=meta.get('count'); cursor=meta.get('next_cursor')
    if isinstance(total,bool) or not isinstance(total,int) or total<0: raise BadPage('invalid_meta_count')
    if task['expected'] is not None and total!=task['expected']: raise BadPage('api_total_changed_during_pagination')
    if cursor is not None and not isinstance(cursor,str): raise BadPage('invalid_cursor')
    if cursor==task['cursor']: raise BadPage('repeated_cursor')
    if not results and cursor: raise BadPage('empty_page_nonterminal_cursor')
    target_set=set(p.get('targets',[])); seen=set(); items=[]; edges=[]
    for w in results:
        if not isinstance(w,dict) or not nid(w.get('id')): raise BadPage('invalid_result_work_id')
        wid=nid(w['id']); year=yr(w.get('publication_year'))
        if wid in seen: raise BadPage('duplicate_work_within_page')
        seen.add(wid)
        if conn.execute('SELECT 1 FROM items WHERE task=? AND id=?',(task['id'],wid)).fetchone():
            raise BadPage('duplicate_work_across_pages_or_live_graph_drift')
        if year is not None:
            previous = {
                r["year"]
                for r in conn.execute("""
                    SELECT DISTINCT i.year
                    FROM items i
                    JOIN tasks t ON t.id = i.task
                    WHERE i.id = ?
                    AND i.year IS NOT NULL
                    AND t.status <> 'superseded'
                """, (wid,))
            }
            if previous and previous!={year}: raise BadPage('publication_year_conflict_across_queries')
        author_ids=[]
        if task['kind']=='citations':
            if year is None or not WINDOW_LO<=year<=WINDOW_HI: raise BadPage('invalid_citing_year')
            refs=w.get('referenced_works'); auth=w.get('authorships')
            if not isinstance(refs,list) or not isinstance(auth,list): raise BadPage('missing_references_or_authorships')
            if any(nid(ref) is None for ref in refs): raise BadPage('invalid_reference_id')
            matched={canonical(ref) for ref in refs}&target_set
            if not matched: raise BadPage('no_canonical_target_overlap_requires_review')
            for slot in auth:
                a=slot.get('author') if isinstance(slot,dict) else None
                aid=nid(a.get('id'),'A') if isinstance(a,dict) else None
                if aid: author_ids.append(aid)
            edges.extend((task['id'],wid,cited,year) for cited in sorted(matched))
        items.append((task['id'],wid,year,json.dumps(sorted(set(author_ids)))))
    collected=conn.execute('SELECT COUNT(*) AS n FROM items WHERE task=?',(task['id'],)).fetchone()['n']+len(items)
    if collected>total: raise BadPage('more_results_than_meta_count')
    if cursor is None and collected!=total: raise BadPage('terminal_page_count_mismatch')
    return total,cursor,items,edges

def restart_drifted(task_id):
    n=conn.execute('SELECT COUNT(*) AS n FROM query_restarts_v2 WHERE task=?',(task_id,)).fetchone()['n']
    if n>=2: return False
    with conn:
        r=conn.execute('INSERT INTO query_restarts_v2(task,reason,at) VALUES(?,?,?)',(task_id,'api_total_changed_during_pagination',now()))
        rid=r.lastrowid
        conn.execute('INSERT INTO retired_items_v2 SELECT ?,task,id,year,author_ids FROM items WHERE task=?',(rid,task_id))
        conn.execute('INSERT INTO retired_edges_v2 SELECT ?,task,citing,cited,year FROM edges WHERE task=?',(rid,task_id))
        conn.execute('DELETE FROM items WHERE task=?',(task_id,))
        conn.execute('DELETE FROM edges WHERE task=?',(task_id,))
        conn.execute("UPDATE tasks SET status='pending',cursor='*',expected=NULL,pages=0,issue='',started=NULL,finished=NULL WHERE id=?",(task_id,))
    return True

def run_task(task_id):
    while True:
        task=conn.execute('SELECT * FROM tasks WHERE id=?',(task_id,)).fetchone()
        if task['status']=='complete': return
        if task['status']=='blocked':
            if task['issue']=='api_total_changed_during_pagination' and restart_drifted(task_id): continue
            raise StopCollection('Query blocked: '+task['issue']+'. See query_progress.csv.')
        if task['status']=='superseded': raise StopCollection('Refusing to run a superseded task.')
        p=json.loads(task['payload'])
        if task['kind']=='portfolio':
            filt='author.id:'+p['author_id']; select='id,publication_year'
        else:
            filt='cites:'+'|'.join(p['targets'])+f',publication_year:{WINDOW_LO}-{WINDOW_HI}'
            select='id,publication_year,referenced_works,authorships'
        body=request_json('works',{'filter':filt,'select':select,'per_page':PER_PAGE,'cursor':task['cursor'],'corpus':CORPUS})
        try:
            total,cursor,items,edges=validate_page(task,body)
        except BadPage as exc:
            with conn: conn.execute("UPDATE tasks SET status='blocked',issue=? WHERE id=?",(str(exc),task_id))
            if str(exc)=='api_total_changed_during_pagination' and restart_drifted(task_id): continue
            raise StopCollection('Query blocked: '+str(exc)+'. See query_progress.csv.')
        with conn:
            conn.executemany('INSERT INTO items VALUES(?,?,?,?)',items)
            conn.executemany('INSERT OR IGNORE INTO edges VALUES(?,?,?,?)',edges)
            conn.execute('UPDATE tasks SET expected=?,cursor=?,pages=pages+1,status=?,started=COALESCE(started,?),finished=? WHERE id=?',
                (total,cursor or '', 'complete' if cursor is None else 'running',now(),now() if cursor is None else None,task_id))

## 4. Assemble portfolios, resolve IDs and collect canonical citation batches

A refreshed portfolio replaces its legacy counterpart for this citation collection only. Reference award works remain known targets and are explicitly flagged if absent from the portfolio. Unresolved IDs prevent main counts for affected authors; they do not block everyone else after the ID audit finishes.

In [11]:
def raw_portfolios():
    states={r['id']:r['status'] for r in conn.execute('SELECT id,status FROM tasks')}
    ready=all(states[t]=='complete' for t in portfolio_tasks.values())
    mapping={}; rows=[]
    for a in master4.itertuples(index=False):
        aid=a.author_id
        refreshed=aid in portfolio_tasks and states[portfolio_tasks[aid]]=='complete'
        ids={r['id'] for r in conn.execute('SELECT id FROM items WHERE task=?',(portfolio_tasks[aid],))} if refreshed else set(legacy.get(aid,set()))
        missing=expected_refs[aid]-ids
        mapping[aid]=ids|expected_refs[aid]
        rows.append({'author_id':aid,'portfolio_source':'refreshed_api_portfolio' if refreshed else 'legacy_cache_unverified',
            'portfolio_api_pagination_complete':refreshed,'n_reference_works_added':len(missing),
            'reference_work_missing_from_portfolio_query':bool(missing),
            'citation_portfolio_identity_independently_verified':False})
    return mapping,pd.DataFrame(rows),ready

def canonical_plan():
    raw,audit,portfolios_ready=raw_portfolios()
    states={r['old_id']:r['status'] for r in conn.execute('SELECT old_id,status FROM work_aliases_v2')}
    ids=set().union(*raw.values())
    ready=portfolios_ready and all(states.get(w) in ('resolved','unresolved_404') for w in ids)
    reload_aliases()
    mapping={}; unresolved={}; merged={}
    for aid,targets in raw.items():
        unresolved[aid]={w for w in targets if states.get(w)!='resolved'}
        mapping[aid]={canonical(w) for w in targets if states.get(w)=='resolved'}
        merged[aid]=sum(states.get(w)=='resolved' and canonical(w)!=w for w in targets)
    audit['n_unresolved_target_ids']=audit.author_id.map(lambda a:len(unresolved[a]))
    audit['n_canonical_target_works']=audit.author_id.map(lambda a:len(mapping[a]))
    audit['n_merged_target_ids']=audit.author_id.map(merged)
    plan=[]
    if ready:
        targets=sorted(set().union(*mapping.values()))
        fingerprint=stable_hash(sorted((a,sorted(ws)) for a,ws in mapping.items()))
        saved=conn.execute("SELECT value FROM metadata WHERE key='canonical_target_plan_hash_v2'").fetchone()
        if saved and saved['value']!=fingerprint: raise RuntimeError('Canonical plan changed after locking; do not mix snapshots.')
        with conn: conn.execute("INSERT OR IGNORE INTO metadata VALUES('canonical_target_plan_hash_v2',?)",(fingerprint,))
        for start in range(0,len(targets),BATCH_SIZE):
            batch=targets[start:start+BATCH_SIZE]
            payload={'targets':batch,'year_lo':WINDOW_LO,'year_hi':WINDOW_HI,'alias_policy':ALIAS_POLICY}
            plan.append((add_task('citations',payload),batch))
    with conn:
        conn.execute('DELETE FROM active_citation_tasks_v2')
        conn.executemany('INSERT INTO active_citation_tasks_v2 VALUES(?)',[(tid,) for tid,_ in plan])
    return mapping,unresolved,audit,plan,ready

try:
    if RUN_API_COLLECTION:
        session.headers['Authorization']='Bearer '+get_api_key()
        try:
            for tid in portfolio_tasks.values(): run_task(tid)
            raw,_,ready=raw_portfolios()
            if not ready: raise StopCollection('Portfolio refresh incomplete.')
            print('Auditing target IDs before citation queries. Capped runs resume automatically.')
            resolve_targets(set().union(*raw.values()))
            mapping,unresolved_by_author,portfolio_audit,citation_plan,plan_ready=canonical_plan()
            print('Canonical citation batches:',len(citation_plan),'| Pagination adds requests.')
            for tid,_ in citation_plan: run_task(tid)
            stop_reason='All resolvable target queries complete; inspect unresolved-ID coverage.'
        finally: session.headers.pop('Authorization',None)
    else: stop_reason='Preview only; no live requests.'
except StopCollection as exc:
    stop_reason=str(exc)
finally:
    session.headers.pop('Authorization',None)
mapping,unresolved_by_author,portfolio_audit,citation_plan,plan_ready=canonical_plan()
print('Collection status:',stop_reason)
print('HTTP attempts this run:',requests_this_run)
print('No key was saved. Do not delete the checkpoint; the next run resumes it.')

Auditing target IDs before citation queries. Capped runs resume automatically.
Canonical citation batches: 3386 | Pagination adds requests.
Collection status: Query blocked: publication_year_conflict_across_queries. See query_progress.csv.
HTTP attempts this run: 4422
No key was saved. Do not delete the checkpoint; the next run resumes it.


In [12]:
raw, _, _ = raw_portfolios()
targets = set().union(*raw.values())

states = {
    r["old_id"]: r["status"]
    for r in conn.execute(
        "SELECT old_id, status FROM work_aliases_v2"
    )
}

resolved = sum(states.get(w) == "resolved" for w in targets)
unavailable = sum(states.get(w) == "unresolved_404" for w in targets)
pending_single = sum(states.get(w) == "pending_single" for w in targets)
unchecked = sum(w not in states for w in targets)

print("Unique target IDs:", len(targets))
print("Resolved:", resolved)
print("Unresolved 404s:", unavailable)
print("Pending individual lookups:", pending_single)
print("Not yet batch-checked:", unchecked)
print("Minimum remaining batch lookups:",
      math.ceil(unchecked / BATCH_SIZE))

Unique target IDs: 369416
Resolved: 366516
Unresolved 404s: 2900
Pending individual lookups: 0
Not yet batch-checked: 0
Minimum remaining batch lookups: 0


## 5. Aggregate only active canonical edges

Main counts stay missing until every planned batch for an author is complete AND none of that author's target IDs is unresolved. Observed-so-far counts are explicitly partial; zeros there do not mean completed collection. Same canonical target aliases are counted once per author.

In [13]:
statuses={r['id']:r['status'] for r in conn.execute('SELECT id,status FROM tasks')}
work_task={w:tid for tid,ws in citation_plan for w in ws}
work_authors=defaultdict(set)
for aid,ws in mapping.items():
    for w in ws: work_authors[w].add(aid)
edge_counts=Counter()
for r in conn.execute('SELECT e.cited,e.year,COUNT(DISTINCT e.citing) AS n FROM edges e JOIN active_citation_tasks_v2 a ON a.id=e.task GROUP BY e.cited,e.year'):
    for aid in work_authors.get(r['cited'],set()): edge_counts[(aid,r['year'])]+=int(r['n'])
long_rows=[]; wide_rows=[]
for a in master4.itertuples(index=False):
    aid=a.author_id; targets=mapping[aid]; tids={work_task[w] for w in targets if w in work_task}
    queries_done=plan_ready and bool(targets) and all(w in work_task for w in targets) and all(statuses[t]=='complete' for t in tids)
    usable=queries_done and not unresolved_by_author[aid]
    wide={'author_id':aid,'citation_queries_complete_for_resolvable_targets':queries_done,
        'citation_series_usable':usable,'citation_counts_provisional':True,
        'n_citation_target_batches':len(tids),'n_citation_target_batches_complete':sum(statuses[t]=='complete' for t in tids)}
    for k in OFFSETS:
        y=int(a.reference_award_year)+k; observed=int(edge_counts.get((aid,y),0))
        c='cites_m'+str(-k) if k<0 else 'cites_p'+str(k) if k>0 else 'cites_award_year'
        wide[c]=observed if usable else None; wide[c+'_observed_so_far']=observed
        long_rows.append({'author_id':aid,'reference_award_year':int(a.reference_award_year),'relative_year':k,
            'calendar_year':y,'period':'before' if k<0 else 'after' if k>0 else 'award_year',
            'citations_received':observed if usable else None,'citations_received_observed_so_far':observed,
            'citation_queries_complete_for_resolvable_targets':queries_done,'citation_series_usable':usable,
            'citation_counts_provisional':True,'self_citations_included':True,
            'n_unresolved_target_ids':len(unresolved_by_author[aid])})
    wide_rows.append(wide)
long=pd.DataFrame(long_rows); wide=pd.DataFrame(wide_rows)
cite_cols=['cites_m'+str(-k) if k<0 else 'cites_p'+str(k) if k>0 else 'cites_award_year' for k in OFFSETS]
for c in cite_cols+[c+'_observed_so_far' for c in cite_cols]: wide[c]=wide[c].astype('Int64')
for c in ['citations_received','citations_received_observed_so_far']: long[c]=long[c].astype('Int64')
wide['cites_before_5y']=wide[['cites_m'+str(i) for i in range(1,6)]].sum(axis=1,min_count=5).astype('Int64')
wide['cites_after_5y']=wide[['cites_p'+str(i) for i in range(1,6)]].sum(axis=1,min_count=5).astype('Int64')
wide['cites_total_11y']=wide[cite_cols].sum(axis=1,min_count=11).astype('Int64')
collisions=(set(wide)|set(portfolio_audit))&(set(master4)-{'author_id'})
if collisions: raise RuntimeError('Unexpected overlapping output columns: '+repr(collisions))
master5=master4.merge(portfolio_audit,on='author_id',validate='one_to_one').merge(wide,on='author_id',validate='one_to_one')
assert len(master5)==len(master4) and master5.author_id.is_unique
assert len(long)==11*len(master4) and not long[['author_id','relative_year']].duplicated().any()
assert long.calendar_year.eq(long.reference_award_year+long.relative_year).all()
assert long.loc[~long.citation_series_usable,'citations_received'].isna().all()
assert long.citations_received.dropna().ge(0).all()
for k,c in zip(OFFSETS,cite_cols):
    pd.testing.assert_series_equal(master5.set_index('author_id')[c].sort_index(),
        long.loc[long.relative_year.eq(k)].set_index('author_id').citations_received.sort_index(),check_names=False)
usable=wide.citation_series_usable
assert (wide.loc[usable,'cites_before_5y']+wide.loc[usable,'cites_award_year']+wide.loc[usable,'cites_after_5y']).eq(wide.loc[usable,'cites_total_11y']).all()
display(master5[['author_id','author_name','reference_award_year','citation_series_usable',*cite_cols]].head(15))

,author_id,author_name,reference_award_year,citation_series_usable,cites_m5,cites_m4,cites_m3,cites_m2,cites_m1,cites_award_year,cites_p1,cites_p2,cites_p3,cites_p4,cites_p5
0,A5000009841,Ana Klimovic,2014,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,A5000102893,David Akers,2009,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,A5000107991,Michael Vrable,2008,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,A5000163896,Elisa D. Mekler,2016,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,A5000167011,I. Scott MacKenzie,2008,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
5,A5000170074,Alexandru Agache,2017,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
6,A5000245150,Alexander J. Smola,2014,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
7,A5000246567,Matthew L. Lee,2014,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
8,A5000294737,Philipp Schwartenbeck,2016,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
9,A5000308912,Jaechang Nam,2013,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


## 6. Export a new progress snapshot on every run

Check the summary, query progress, alias audit and unresolved-target coverage. Pending counts are not zeros. Snapshots remain provisional even after API queries complete because legacy portfolio completeness and author-ID disambiguation are not independently verified. Citing-author IDs remain in the SQLite checkpoint for step 6; missing IDs must not be assumed non-self-citations.

In [14]:
progress=pd.read_sql_query('SELECT id,kind,status,expected,pages,issue,started,finished FROM tasks ORDER BY kind,id',conn)
alias_audit=pd.read_sql_query('SELECT * FROM work_aliases_v2 ORDER BY old_id',conn)
restart_audit=pd.read_sql_query('SELECT * FROM query_restarts_v2 ORDER BY restart_id',conn)
usage=pd.read_sql_query('SELECT utc_day,COUNT(*) AS http_attempts FROM request_ledger GROUP BY utc_day',conn)
unresolved_audit=pd.DataFrame([{'author_id':a,'old_target_id':w,'reason':'unresolved_404_or_alias_audit_pending'}
    for a,ws in unresolved_by_author.items() for w in sorted(ws)],columns=['author_id','old_target_id','reason'])
active_edge_n=conn.execute('SELECT COUNT(*) AS n FROM edges e JOIN active_citation_tasks_v2 a ON a.id=e.task').fetchone()['n']
summary_values={'authors_retained':len(master5),'author_year_rows':len(long),
    'portfolio_refresh_queries_planned':len(portfolio_tasks),'portfolio_refresh_queries_complete':sum(statuses[t]=='complete' for t in portfolio_tasks.values()),
    'target_ids_resolved':int(alias_audit.status.eq('resolved').sum()),'target_ids_unresolved_404':int(alias_audit.status.eq('unresolved_404').sum()),
    'target_ids_pending_single_lookup':int(alias_audit.status.eq('pending_single').sum()),
    'canonical_citation_batches_planned':len(citation_plan),'canonical_citation_batches_complete':sum(statuses[t]=='complete' for t,_ in citation_plan),
    'authors_with_usable_citation_series':int(wide.citation_series_usable.sum()),
    'authors_missing_main_citation_series':int((~wide.citation_series_usable).sum()),
    'authors_with_unresolved_target_ids':int(portfolio_audit.n_unresolved_target_ids.gt(0).sum()),
    'active_canonical_citation_edges_collected':active_edge_n,'blocked_queries':int(progress.status.eq('blocked').sum()),
    'http_attempts_this_run':requests_this_run}
summary=pd.DataFrame(summary_values.items(),columns=['measure','value'])
display(summary)
SNAPSHOT_DIR=ROOT/'conf_paper/conf_data/derived'/COLLECTION_VERSION/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SNAPSHOT_DIR.mkdir(parents=True,exist_ok=False)
tables={'author_yearly_citations.csv':master5,'yearly_citations_long.csv':long,'portfolio_coverage.csv':portfolio_audit,
    'query_progress.csv':progress,'work_alias_audit.csv':alias_audit,'unresolved_target_coverage.csv':unresolved_audit,
    'query_restart_audit.csv':restart_audit,'local_request_usage.csv':usage,'step5_summary.csv':summary}
for name,frame in tables.items(): frame.to_csv(SNAPSHOT_DIR/name,index=False)
conn.commit()
meta={'created_at_utc':now(),'legacy_checkpoint_identity':identity,'alias_policy':ALIAS_POLICY,
    'output_sha256':{p.name:sha(p) for p in SNAPSHOT_DIR.iterdir() if p.is_file()},'stop_reason':stop_reason,
    'all_resolvable_citation_batches_complete':bool(plan_ready and all(statuses[t]=='complete' for t,_ in citation_plan)),
    'all_authors_have_usable_citation_series':bool(wide.citation_series_usable.all()),'credentials_saved':False,'upstream_files_modified':False,
    'definition':'distinct citing-work/canonical-cited-work edges binned by citing publication year; self-citations included',
    'limitations':['Unrefreshed legacy portfolios remain unverified.','404 targets are missing coverage, not zero citations.',
        'Current graph, not a historical index snapshot.','Canonical aliases reflect collection-time lookups, not immutable global identity.',
        'Old citation tasks are superseded and excluded.','Missing citing-author IDs do not establish non-self-citation.']}
(SNAPSHOT_DIR/'manifest.json').write_text(json.dumps(meta,indent=2),encoding='utf-8')
print('Saved progress snapshot:',SNAPSHOT_DIR)
print('Next run resumes this same checkpoint. Main counts stay missing until the required coverage checks pass.')

,measure,value
0,authors_retained,2710
1,author_year_rows,29810
2,portfolio_refresh_queries_planned,125
3,portfolio_refresh_queries_complete,125
4,target_ids_resolved,366516
5,target_ids_unresolved_404,2900
6,target_ids_pending_single_lookup,0
7,canonical_citation_batches_planned,3386
8,canonical_citation_batches_complete,1
9,authors_with_usable_citation_series,0


Saved progress snapshot: B:\Semester 4 UU\thesis-best-paper-trajectories\conf_paper\conf_data\derived\yearly_citations_v1\20261117T165057408090Z
Next run resumes this same checkpoint. Main counts stay missing until the required coverage checks pass.


In [15]:
for p in ROOT.rglob("query_progress.csv"):
    print("File location:", p.resolve())
    progress = pd.read_csv(p)
    display(progress.loc[progress["status"].eq("blocked")])

File location: B:\Semester 4 UU\thesis-best-paper-trajectories\conf_paper\conf_data\derived\yearly_citations_v1\20261117T165057408090Z\query_progress.csv


,id,kind,status,expected,pages,issue,started,finished
6029,dac4fe387e08d98761f2720ab9b4845f46310a3bc4c850...,citations,blocked,NaN,0,publication_year_conflict_across_queries,NaN,NaN


In [16]:
task = conn.execute("""
    SELECT * FROM tasks
    WHERE status='blocked'
      AND issue='publication_year_conflict_across_queries'
    LIMIT 1
""").fetchone()

if task is None:
    raise RuntimeError("No matching blocked task found.")

payload = json.loads(task["payload"])
params = {
    "filter": (
        "cites:" + "|".join(payload["targets"])
        + f",publication_year:{WINDOW_LO}-{WINDOW_HI}"
    ),
    "select": "id,publication_year",
    "per_page": PER_PAGE,
    "cursor": task["cursor"],
    "corpus": CORPUS,
    "api_key": get_api_key(),
}

try:
    body = request_json("works", params)
finally:
    params.pop("api_key", None)

conflicts = []
for work in body.get("results", []):
    wid = nid(work.get("id"))
    incoming_year = yr(work.get("publication_year"))
    if incoming_year is None:
        continue

    previous = conn.execute("""
        SELECT i.year, t.kind, t.status, t.id AS source_task
        FROM items i JOIN tasks t ON t.id=i.task
        WHERE i.id=? AND i.year IS NOT NULL AND i.year<>?
    """, (wid, incoming_year)).fetchall()

    for row in previous:
        conflicts.append({
            "work_id": wid,
            "incoming_year": incoming_year,
            "saved_year": row["year"],
            "source_kind": row["kind"],
            "source_status": row["status"],
            "source_task": row["source_task"],
        })

display(pd.DataFrame(conflicts))

,work_id,incoming_year,saved_year,source_kind,source_status,source_task
0,W83339351,2004,2018,citations,superseded,74b1f8e8c025ca4ba9e3efa352f81d6c4afd574b1680df...
